# Quantitative comparison: Online STDR, STS-Miner, and STBFM

This notebook runs a quantitative experiment on the preprocessed TSMC2014 NYC data. The online STDR miner is updated for every event instance, and STS-Miner/STBFM are rerun on every prefix using the same binary-search threshold tuning procedure.

`CHECKPOINT_EVERY` is kept as the snapshot cadence. Rule CSVs are written only at checkpoint event counts; timing rows are written for every evaluated event instance.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from math import ceil
from pathlib import Path
import sys
from time import perf_counter
from typing import Callable

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from spatio_temporal_network import (
    LearningParameters,
    OnlineSTDRMiner,
    initialize_network_from_events,
)
from sts_miner import STSMiner, add_local_meter_coordinates, patterns_to_frame as sts_rules_to_frame
from stbfm import STBFM, patterns_to_frame as stbfm_rules_to_frame


## Configuration

`CHECKPOINT_EVERY` controls how often rule snapshots are written to disk. Mining times are collected for every event prefix up to `MAX_EVENTS`.


In [2]:
from math import log
DATA_PATH = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Quantitative_experiment_comparison"
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID

EVENT_TYPE_COUNTS = (15,20)
MAX_EVENTS = 5000
CHECKPOINT_EVERY = 200
CHECKPOINTS = tuple(range(CHECKPOINT_EVERY, MAX_EVENTS + 1, CHECKPOINT_EVERY))

GRID_SHAPE = (10, 10)
ONLINE_SPATIAL_THRESHOLD_METERS = 10_000.0
ONLINE_ETA = 1.0
ONLINE_TAU_ACTIVATION = 60.0
ONLINE_TAU_REFRACTORY = 12
ONLINE_THETAS = (0.1, 0.15, 0.2, 0.25)
ONLINE_LAMBDA_DECAY = -log(0.10) / 120.0

OFFLINE_WINDOW_MINUTES = -log(0.10) / ONLINE_LAMBDA_DECAY

MAX_RULE_LENGTH = 2
MAX_SNAPSHOT_RULES = None
RELATIVE_RULE_COUNT_TOLERANCE = 0.15

R_CANDIDATES_METERS = (10_000.0,)
T_CANDIDATES_MINUTES = (120,)
BINARY_SEARCH_ITERATIONS = 10
STOP_AFTER_FIRST_TOLERANCE_MATCH = True
STS_THRESHOLD_LOW = 1.0
STS_THRESHOLD_HIGH_LIMIT = 1_000_000.0
STBFM_THRESHOLD_LOW = 0.0
STBFM_THRESHOLD_HIGH = 1.0

RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR


PosixPath('/Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Quantitative_experiment_comparison/run_20260602_113111')

In [3]:
def rule_count_tolerance(target_count: int) -> int:
    if target_count == 0:
        return 0
    return max(1, ceil(target_count * RELATIVE_RULE_COUNT_TOLERANCE))


def within_tolerance(rule_count: int, target_count: int) -> bool:
    return abs(rule_count - target_count) <= rule_count_tolerance(target_count)


def load_selected_events(event_type_count: int) -> tuple[pd.DataFrame, tuple[str, ...]]:
    raw_events = pd.read_csv(DATA_PATH).sort_values("timestamp", ignore_index=True)
    selected_types = tuple(raw_events["Event_type"].value_counts().head(event_type_count).index)
    selected_events = raw_events[raw_events["Event_type"].isin(selected_types)].head(MAX_EVENTS).copy()
    selected_events = selected_events.sort_values("timestamp", ignore_index=True)
    if len(selected_events) < MAX_EVENTS:
        raise ValueError(
            f"Only {len(selected_events)} events are available for the top {event_type_count} event types."
        )
    return selected_events, selected_types


def make_online_parameters(events: pd.DataFrame, online_theta: float) -> LearningParameters:
    observation_window_minutes = float(events["timestamp"].iloc[-1] - events["timestamp"].iloc[0])
    if observation_window_minutes <= 0:
        raise ValueError("The selected stream must cover a positive time span.")
    return LearningParameters(
        eta=ONLINE_ETA,
        lambda_decay=ONLINE_LAMBDA_DECAY,
        tau_activation=ONLINE_TAU_ACTIVATION,
        tau_refractory=ONLINE_TAU_REFRACTORY,
        theta=online_theta,
    )


def relative_path_or_none(path: Path | None) -> str | None:
    if path is None:
        return None
    return str(path.relative_to(PROJECT_ROOT))


def merged_event_type_rule_count(rules: list) -> int:
    return len({tuple(rule.event_type_rule) for rule in rules})


In [4]:
def run_online_per_event(
    events: pd.DataFrame,
    output_dir: Path,
    event_type_count: int,
    online_theta: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    online_dir = output_dir / "online_stdr"
    online_dir.mkdir(parents=True, exist_ok=True)

    parameters = make_online_parameters(events, online_theta)
    network = initialize_network_from_events(
        events,
        grid_shape=GRID_SHAPE,
        spatial_threshold=ONLINE_SPATIAL_THRESHOLD_METERS,
    )
    miner = OnlineSTDRMiner(network, parameters)
    checkpoint_set = set(CHECKPOINTS)

    timing_rows = []
    checkpoint_rows = []
    cumulative_seconds = 0.0
    total_events = len(events)

    for event_index, event in enumerate(events.itertuples(index=False), start=1):
        event_series = pd.Series(event._asdict())
        start = perf_counter()
        miner.process_event(event_series, extract_rules=False)
        rules = miner.extract_rules(
            max_rule_length=MAX_RULE_LENGTH,
            max_rules=MAX_SNAPSHOT_RULES,
        )
        mining_seconds = perf_counter() - start
        cumulative_seconds += mining_seconds

        merge_start = perf_counter()
        merged_rule_count = merged_event_type_rule_count(rules)
        merge_seconds = perf_counter() - merge_start

        is_checkpoint = event_index in checkpoint_set
        rules_path = None
        if is_checkpoint:
            stem = f"online_stdr_{event_index:06d}"
            rules_path = online_dir / f"{stem}_rules.csv"
            miner.rules_frame(rules).to_csv(rules_path, index=False)

        row = {
            "event_type_count": event_type_count,
            "online_theta": online_theta,
            "algorithm": "online_stdr",
            "event_count": event_index,
            "is_checkpoint": is_checkpoint,
            "rule_count": merged_rule_count,
            "raw_rule_count": len(rules),
            "target_rule_count": merged_rule_count,
            "significant_synapse_count": len(miner.significant_synapses),
            "mining_seconds": mining_seconds,
            "merge_seconds": merge_seconds,
            "cumulative_seconds": cumulative_seconds,
            "rules_path": relative_path_or_none(rules_path),
            "spatial_radius": None,
            "temporal_window": None,
            "threshold": online_theta,
            "attempt_count": None,
            "tuning_seconds": mining_seconds,
            "within_tolerance": True,
        }
        timing_rows.append(row)
        if is_checkpoint:
            checkpoint_rows.append(row.copy())
            print(
                f"      online_stdr event={event_index}: "
                f"cumulative={cumulative_seconds:.6f}s, "
                f"merged_rules={merged_rule_count}, raw_rules={len(rules)}, "
                f"merge={merge_seconds:.6f}s, significant_synapses={len(miner.significant_synapses)}"
            )
        elif event_index == total_events:
            print(
                f"      online_stdr final event={event_index}: "
                f"cumulative={cumulative_seconds:.6f}s, "
                f"merged_rules={merged_rule_count}, raw_rules={len(rules)}, "
                f"merge={merge_seconds:.6f}s, significant_synapses={len(miner.significant_synapses)}"
            )

    return pd.DataFrame(timing_rows), pd.DataFrame(checkpoint_rows)


In [5]:
@dataclass
class TrialResult:
    algorithm: str
    event_count: int
    spatial_radius: float
    temporal_window: float
    threshold: float
    rule_count: int
    mining_seconds: float
    rules: list


def run_sts_trial(events: pd.DataFrame, spatial_radius: float, temporal_window: float, threshold: float) -> tuple[list, float]:
    start = perf_counter()
    miner = STSMiner(
        events,
        spatial_radius=spatial_radius,
        temporal_window=temporal_window,
        min_sequence_index=threshold,
        max_length=MAX_RULE_LENGTH,
    )
    rules = miner.mine()
    return rules, perf_counter() - start


def run_stbfm_trial(events: pd.DataFrame, spatial_radius: float, temporal_window: float, threshold: float) -> tuple[list, float]:
    start = perf_counter()
    miner = STBFM(
        events,
        spatial_radius=spatial_radius,
        temporal_window=temporal_window,
        min_participation_index=threshold,
        max_length=MAX_RULE_LENGTH,
        threshold_inclusive=False,
    )
    rules = miner.mine()
    return rules, perf_counter() - start


def score_trial(trial: TrialResult, target_count: int) -> tuple[int, float, float, float]:
    return (
        abs(trial.rule_count - target_count),
        trial.mining_seconds,
        trial.spatial_radius,
        trial.temporal_window,
    )


In [6]:
def tune_threshold_for_radius_window(
    *,
    algorithm: str,
    events: pd.DataFrame,
    event_count: int,
    target_count: int,
    spatial_radius: float,
    temporal_window: float,
    run_trial: Callable[[pd.DataFrame, float, float, float], tuple[list, float]],
    threshold_low: float,
    threshold_high: float,
    expandable_high: bool,
) -> tuple[TrialResult, list[dict]]:
    attempts = []

    def evaluate(threshold: float, phase: str) -> TrialResult:
        rules, seconds = run_trial(events, spatial_radius, temporal_window, threshold)
        trial = TrialResult(
            algorithm=algorithm,
            event_count=event_count,
            spatial_radius=spatial_radius,
            temporal_window=temporal_window,
            threshold=threshold,
            rule_count=len(rules),
            mining_seconds=seconds,
            rules=rules,
        )
        attempts.append({
            "algorithm": algorithm,
            "event_count": event_count,
            "target_rule_count": target_count,
            "spatial_radius": spatial_radius,
            "temporal_window": temporal_window,
            "threshold": threshold,
            "rule_count": trial.rule_count,
            "mining_seconds": trial.mining_seconds,
            "phase": phase,
        })
        return trial

    low_trial = evaluate(threshold_low, "low")
    best = low_trial
    low = threshold_low
    high = threshold_high

    high_trial = evaluate(high, "high")
    if score_trial(high_trial, target_count) < score_trial(best, target_count):
        best = high_trial

    while expandable_high and high_trial.rule_count > target_count and high < STS_THRESHOLD_HIGH_LIMIT:
        low = high
        high = min(high * 2.0, STS_THRESHOLD_HIGH_LIMIT)
        high_trial = evaluate(high, "expand_high")
        if score_trial(high_trial, target_count) < score_trial(best, target_count):
            best = high_trial

    for iteration in range(BINARY_SEARCH_ITERATIONS):
        midpoint = (low + high) / 2.0
        trial = evaluate(midpoint, f"binary_{iteration + 1}")
        if score_trial(trial, target_count) < score_trial(best, target_count):
            best = trial

        if trial.rule_count > target_count:
            low = midpoint
        else:
            high = midpoint

        if within_tolerance(trial.rule_count, target_count):
            break

    return best, attempts


In [7]:
def tune_offline_algorithm_for_prefix(
    *,
    algorithm: str,
    events: pd.DataFrame,
    event_count: int,
    target_count: int,
) -> tuple[TrialResult, list[dict]]:
    if algorithm == "sts_miner":
        run_trial = run_sts_trial
        threshold_low = STS_THRESHOLD_LOW
        threshold_high = max(STS_THRESHOLD_LOW * 2.0, 2.0)
        expandable_high = True
    elif algorithm == "stbfm":
        run_trial = run_stbfm_trial
        threshold_low = STBFM_THRESHOLD_LOW
        threshold_high = STBFM_THRESHOLD_HIGH
        expandable_high = False
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    all_attempts = []
    best: TrialResult | None = None
    for spatial_radius in R_CANDIDATES_METERS:
        for temporal_window in T_CANDIDATES_MINUTES:
            candidate, attempts = tune_threshold_for_radius_window(
                algorithm=algorithm,
                events=events,
                event_count=event_count,
                target_count=target_count,
                spatial_radius=spatial_radius,
                temporal_window=temporal_window,
                run_trial=run_trial,
                threshold_low=threshold_low,
                threshold_high=threshold_high,
                expandable_high=expandable_high,
            )
            all_attempts.extend(attempts)
            if best is None or score_trial(candidate, target_count) < score_trial(best, target_count):
                best = candidate
            if STOP_AFTER_FIRST_TOLERANCE_MATCH and within_tolerance(best.rule_count, target_count):
                break
        if STOP_AFTER_FIRST_TOLERANCE_MATCH and best is not None and within_tolerance(best.rule_count, target_count):
            break

    if best is None:
        raise RuntimeError(f"No trial was run for {algorithm} at {event_count} events.")
    return best, all_attempts


def write_offline_rules(algorithm: str, rules: list, rules_path: Path) -> None:
    rules_path.parent.mkdir(parents=True, exist_ok=True)
    if algorithm == "sts_miner":
        sts_rules_to_frame(rules).to_csv(rules_path, index=False)
    elif algorithm == "stbfm":
        stbfm_rules_to_frame(rules).to_csv(rules_path, index=False)
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")


In [8]:
def run_offline_per_event(
    *,
    algorithm: str,
    events: pd.DataFrame,
    output_dir: Path,
    event_type_count: int,
    online_theta: float,
    online_counts: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    algorithm_dir = output_dir / algorithm
    algorithm_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_set = set(CHECKPOINTS)

    timing_rows = []
    checkpoint_rows = []
    attempts_rows = []
    cumulative_seconds = 0.0
    cumulative_tuning_seconds = 0.0
    total_events = len(events)

    for event_count in range(1, len(events) + 1):
        current_time = float(events["timestamp"].iloc[event_count - 1])
        window_start_time = current_time - OFFLINE_WINDOW_MINUTES
        window_events = events[
            (events["timestamp"] >= window_start_time)
            & (events["timestamp"] <= current_time)
        ].copy()
        window_events = add_local_meter_coordinates(window_events)
        target_count = int(online_counts.loc[event_count])
        best, attempts = tune_offline_algorithm_for_prefix(
            algorithm=algorithm,
            events=window_events,
            event_count=event_count,
            target_count=target_count,
        )
        tuning_seconds = sum(float(attempt["mining_seconds"]) for attempt in attempts)
        cumulative_seconds += best.mining_seconds
        cumulative_tuning_seconds += tuning_seconds

        is_checkpoint = event_count in checkpoint_set
        rules_path = None
        if is_checkpoint:
            stem = f"{algorithm}_{event_count:06d}"
            rules_path = algorithm_dir / f"{stem}_rules.csv"
            write_offline_rules(algorithm, best.rules, rules_path)

        for attempt in attempts:
            attempts_rows.append({
                "event_type_count": event_type_count,
                "online_theta": online_theta,
                "window_start_time": window_start_time,
                "window_end_time": current_time,
                "window_event_count": len(window_events),
                **attempt,
            })

        row = {
            "event_type_count": event_type_count,
            "online_theta": online_theta,
            "algorithm": algorithm,
            "event_count": event_count,
            "is_checkpoint": is_checkpoint,
            "window_start_time": window_start_time,
            "window_end_time": current_time,
            "window_event_count": len(window_events),
            "rule_count": best.rule_count,
            "target_rule_count": target_count,
            "significant_synapse_count": None,
            "mining_seconds": best.mining_seconds,
            "cumulative_seconds": cumulative_seconds,
            "rules_path": relative_path_or_none(rules_path),
            "spatial_radius": best.spatial_radius,
            "temporal_window": best.temporal_window,
            "threshold": best.threshold,
            "attempt_count": len(attempts),
            "tuning_seconds": tuning_seconds,
            "within_tolerance": within_tolerance(best.rule_count, target_count),
        }
        timing_rows.append(row)
        if is_checkpoint:
            checkpoint_rows.append(row.copy())
            print(
                f"      {algorithm} event={event_count}: "
                f"selected_cumulative={cumulative_seconds:.6f}s, "
                f"tuning_cumulative={cumulative_tuning_seconds:.6f}s, "
                f"rules={best.rule_count}, target={target_count}, "
                f"window_events={len(window_events)}, attempts={len(attempts)}"
            )
        elif event_count == total_events:
            print(
                f"      {algorithm} final event={event_count}: "
                f"selected_cumulative={cumulative_seconds:.6f}s, "
                f"tuning_cumulative={cumulative_tuning_seconds:.6f}s, "
                f"rules={best.rule_count}, target={target_count}, "
                f"window_events={len(window_events)}, attempts={len(attempts)}"
            )

    return pd.DataFrame(timing_rows), pd.DataFrame(checkpoint_rows), pd.DataFrame(attempts_rows)


## Run the experiment

This cell evaluates all algorithms at every event prefix. Rule CSV snapshots are saved only at the configured checkpoints.


In [9]:
experiment_start = perf_counter()
all_summary_rows = []
all_checkpoint_rows = []
all_attempt_rows = []
selected_type_rows = []

config_rows = [{
    "run_id": RUN_ID,
    "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "event_type_counts": EVENT_TYPE_COUNTS,
    "online_thetas": ONLINE_THETAS,
    "max_events": MAX_EVENTS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "checkpoints": CHECKPOINTS,
    "grid_shape": GRID_SHAPE,
    "online_spatial_threshold_meters": ONLINE_SPATIAL_THRESHOLD_METERS,
    "online_eta": ONLINE_ETA,
    "online_tau_activation": ONLINE_TAU_ACTIVATION,
    "online_tau_refractory": ONLINE_TAU_REFRACTORY,
    "online_lambda_decay": ONLINE_LAMBDA_DECAY,
    "offline_window_minutes": OFFLINE_WINDOW_MINUTES,
    "max_rule_length": MAX_RULE_LENGTH,
    "max_snapshot_rules": MAX_SNAPSHOT_RULES,
    "online_rule_count_for_tuning": "merged_event_type_rule_count",
    "online_merge_time_excluded_from_cumulative": True,
    "relative_rule_count_tolerance": RELATIVE_RULE_COUNT_TOLERANCE,
    "r_candidates_meters": R_CANDIDATES_METERS,
    "t_candidates_minutes": T_CANDIDATES_MINUTES,
    "binary_search_iterations": BINARY_SEARCH_ITERATIONS,
    "stop_after_first_tolerance_match": STOP_AFTER_FIRST_TOLERANCE_MATCH,
}]
pd.DataFrame(config_rows).to_csv(RUN_DIR / "experiment_config.csv", index=False)

for event_type_count in EVENT_TYPE_COUNTS:
    print(f"Running event_type_count={event_type_count}")
    type_dir = RUN_DIR / f"event_types_{event_type_count:02d}"
    type_dir.mkdir(parents=True, exist_ok=True)

    selected_events, selected_types = load_selected_events(event_type_count)
    selected_events.to_csv(type_dir / "selected_events.csv", index=False)
    selected_type_rows.extend(
        {"event_type_count": event_type_count, "rank": rank, "event_type": event_type}
        for rank, event_type in enumerate(selected_types, start=1)
    )

    for online_theta in ONLINE_THETAS:
        theta_label = f"theta_{online_theta:.2f}".replace(".", "_")
        pair_dir = type_dir / theta_label
        pair_dir.mkdir(parents=True, exist_ok=True)
        print(f"  online_theta={online_theta}")

        online_summary, online_checkpoints = run_online_per_event(
            selected_events,
            pair_dir,
            event_type_count,
            online_theta,
        )
        all_summary_rows.extend(online_summary.to_dict("records"))
        all_checkpoint_rows.extend(online_checkpoints.to_dict("records"))
        online_counts = online_summary.set_index("event_count")["rule_count"]

        for algorithm in ("sts_miner", "stbfm"):
            print(f"    {algorithm}: evaluating {len(selected_events)} event prefixes")
            algorithm_summary, algorithm_checkpoints, algorithm_attempts = run_offline_per_event(
                algorithm=algorithm,
                events=selected_events,
                output_dir=pair_dir,
                event_type_count=event_type_count,
                online_theta=online_theta,
                online_counts=online_counts,
            )
            all_summary_rows.extend(algorithm_summary.to_dict("records"))
            all_checkpoint_rows.extend(algorithm_checkpoints.to_dict("records"))
            all_attempt_rows.extend(algorithm_attempts.to_dict("records"))

summary = pd.DataFrame(all_summary_rows)
checkpoint_summary = pd.DataFrame(all_checkpoint_rows)
threshold_attempts = pd.DataFrame(all_attempt_rows)
selected_event_types = pd.DataFrame(selected_type_rows)

summary.to_csv(RUN_DIR / "summary.csv", index=False)
checkpoint_summary.to_csv(RUN_DIR / "checkpoint_summary.csv", index=False)
threshold_attempts.to_csv(RUN_DIR / "threshold_attempts.csv", index=False)
selected_event_types.to_csv(RUN_DIR / "selected_event_types.csv", index=False)

elapsed_seconds = perf_counter() - experiment_start
pd.DataFrame([{"run_id": RUN_ID, "elapsed_seconds": elapsed_seconds}]).to_csv(
    RUN_DIR / "run_metadata.csv",
    index=False,
)

summary.head()


Running event_type_count=15
  online_theta=0.1
      online_stdr event=200: cumulative=1.504769s, merged_rules=157, raw_rules=613, merge=0.000279s, significant_synapses=613
      online_stdr event=400: cumulative=4.145654s, merged_rules=150, raw_rules=759, merge=0.000353s, significant_synapses=759
      online_stdr event=600: cumulative=5.533737s, merged_rules=31, raw_rules=96, merge=0.000044s, significant_synapses=96
      online_stdr event=800: cumulative=6.887706s, merged_rules=147, raw_rules=542, merge=0.000245s, significant_synapses=542
      online_stdr event=1000: cumulative=8.819239s, merged_rules=138, raw_rules=309, merge=0.000164s, significant_synapses=309
      online_stdr event=1200: cumulative=10.161380s, merged_rules=114, raw_rules=237, merge=0.000104s, significant_synapses=237
      online_stdr event=1400: cumulative=11.555111s, merged_rules=2, raw_rules=2, merge=0.000002s, significant_synapses=2
      online_stdr event=1600: cumulative=12.658485s, merged_rules=0, raw_ru

,event_type_count,online_theta,algorithm,event_count,is_checkpoint,rule_count,raw_rule_count,target_rule_count,significant_synapse_count,mining_seconds,...,rules_path,spatial_radius,temporal_window,threshold,attempt_count,tuning_seconds,within_tolerance,window_start_time,window_end_time,window_event_count
0,15,0.1,online_stdr,1,False,0,0.0,0,0.0,0.002836,...,None,NaN,NaN,0.1,NaN,0.002836,True,NaN,NaN,NaN
1,15,0.1,online_stdr,2,False,0,0.0,0,0.0,0.003670,...,None,NaN,NaN,0.1,NaN,0.003670,True,NaN,NaN,NaN
2,15,0.1,online_stdr,3,False,0,0.0,0,0.0,0.002998,...,None,NaN,NaN,0.1,NaN,0.002998,True,NaN,NaN,NaN
3,15,0.1,online_stdr,4,False,0,0.0,0,0.0,0.002806,...,None,NaN,NaN,0.1,NaN,0.002806,True,NaN,NaN,NaN
4,15,0.1,online_stdr,5,False,0,0.0,0,0.0,0.002087,...,None,NaN,NaN,0.1,NaN,0.002087,True,NaN,NaN,NaN
